In [2]:
import arcpy

# 0. 使用目前的 Map 與預設 GDB 環境設定
aprx = arcpy.mp.ArcGISProject("CURRENT")
default_gdb = aprx.defaultGeodatabase
arcpy.env.workspace = default_gdb
arcpy.env.overwriteOutput = True

# --- 參數設定區塊 ---
# 直接使用 ArcGIS Pro 左側內容窗格 (Contents Pane) 的「圖層名稱」
in_points = "k1a_recruit_centroids_2025"      
in_polygons = "rr_1a_2023_coral" # 此圖層會自動套用你設定的 Definition Query
field_name = "on_old_coral"

try:
    print(f"目前使用的預設資料庫: {default_gdb}")
    
    print(f"步驟 1: 檢查並新增 [{field_name}] 欄位...")
    existing_fields = [f.name for f in arcpy.ListFields(in_points)]
    if field_name not in existing_fields:
        arcpy.management.AddField(in_table=in_points, 
                                  field_name=field_name, 
                                  field_type="TEXT", 
                                  field_length=1)

    print("步驟 2: 將所有點的預設值填入 'N'...")
    arcpy.management.CalculateField(in_table=in_points, 
                                    field=field_name, 
                                    expression="'N'", 
                                    expression_type="PYTHON3")

    total_count = int(arcpy.management.GetCount(in_points).getOutput(0))

    print("步驟 3: 尋找與 Polygon 重疊的點 (自動套用介面上的 Definition Query)...")
    # 因為直接使用 Map Layer，不需要額外建立暫存圖層
    arcpy.management.SelectLayerByLocation(in_layer=in_points, 
                                           overlap_type="INTERSECT", 
                                           select_features=in_polygons, 
                                           selection_type="NEW_SELECTION")

    overlap_count = int(arcpy.management.GetCount(in_points).getOutput(0))

    if overlap_count > 0:
        print("步驟 4: 將重疊的點標記為 'Y'...")
        arcpy.management.CalculateField(in_table=in_points, 
                                        field=field_name, 
                                        expression="'Y'", 
                                        expression_type="PYTHON3")

    print("步驟 5: 清除選取狀態...")
    arcpy.management.SelectLayerByAttribute(in_points, "CLEAR_SELECTION")

    if total_count > 0:
        percentage = (overlap_count / total_count) * 100
    else:
        percentage = 0.0

    print("\n" + "="*30)
    print("✅ 處理完成！統計結果如下：")
    print(f"  ▶ 總點數: {total_count}")
    print(f"  ▶ 落在 Polygon 上的點數 (Y): {overlap_count}")
    print(f"  ▶ 未重疊的點數 (N): {total_count - overlap_count}")
    print(f"  ▶ 重疊比例: {percentage:.2f}%")
    print("="*30)

except arcpy.ExecuteError:
    print(f"❌ ArcGIS 執行錯誤: {arcpy.GetMessages(2)}")
except Exception as e:
    print(f"❌ 發生未預期的錯誤: {e}")

目前使用的預設資料庫: D:\ArcGIS\Morpho_Spatial_Demography\MSD_Hawaii.gdb
步驟 1: 檢查並新增 [on_old_coral] 欄位...
步驟 2: 將所有點的預設值填入 'N'...
步驟 3: 尋找與 Polygon 重疊的點 (自動套用介面上的 Definition Query)...
步驟 5: 清除選取狀態...

✅ 處理完成！統計結果如下：
  ▶ 總點數: 648
  ▶ 落在 Polygon 上的點數 (Y): 0
  ▶ 未重疊的點數 (N): 648
  ▶ 重疊比例: 0.00%
